we'll rewrite the products into a standard format.  
LLMs are great at this!

In [10]:
from litellm import completion
from dotenv import load_dotenv
import json
#from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override=True)

True

In [11]:
# LiteLLM allows us to call different LLM providers and models
# using a unified interface through the completion() function.
LITE_MODE = True

In [12]:
username = "ed-donner"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

Loaded 22,000 items
title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Wei

In [14]:
# give every item an id
for index, item in enumerate(items):
    item.id = index

In [15]:
items[2].id

2

In [18]:
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [21]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]

response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(response.choices[0].message.content)
print()

print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")



Title: Schlage F59/613 Andover Interior Half Knob with Deadbolt (Oil Rubbed Bronze)  
Category: Hardware  
Brand: Schlage  
Description: A solid bronze interior knob with integrated deadbolt for added security and style.  
Details: Features easy installation, 4" door prep, and a lifetime mechanical and finish warranty.

Input tokens: 446
Output tokens: 88
Cost: 0.006 cents


In [23]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]

response = completion(messages=messages, model="ollama/qwen2.5-coder:7b", api_base="http://localhost:11434")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

**Title:** Schlage F59 AND 613 Andover Interior Knob with Deadbolt

**Category:** Locks & Security

**Brand:** Schlage

**Description:** The Schlage F59 AND 613 Andover Interior Knob with Deadbolt offers secure and stylish interior entry, featuring precision engineering and a sleek oil rubbed bronze finish.

**Details:** Precision-engineered for security, featuring an oil rubbed bronze finish and easy-to-install design, requiring an F58 handleset for a complete installation.

Input tokens: 419
Output tokens: 110
Cost: 0.000 cents


In [24]:
MODEL = "ollama/qwen2.5-coder:7b"

In [25]:
def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}], "reasoning_effort": "low"}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [26]:
items[0]

<Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only) = $64.3>

In [27]:
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "ollama/qwen2.5-coder:7b", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\\n[\'From the Manufacturer\', \\"When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid\\"]\\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4\\" minimum center to center door prep required for this two pi

In [28]:

def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w", encoding="utf-8") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [30]:
import os

os.makedirs("jsonl", exist_ok=True)

make_file(0, 1000, "jsonl/0_1000.jsonl")

In [31]:
import os
from groq import Groq

groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [ ]:
with open("jsonl/0_1000.jsonl", "rb") as f:
    response = groq.files.create(
        file=f,
        purpose="batch"
    )

response

In [36]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
import requests

response = requests.get("http://localhost:11434/api/tags")
print(response.status_code)
print(response.json())

200
{'models': [{'name': 'qwen2.5-coder:7b', 'model': 'qwen2.5-coder:7b', 'modified_at': '2026-09-25T22:03:50.7287251+05:00', 'size': 4683087561, 'digest': 'dae161e27b0e90dd1856c8bb3209201fd6736d8eb66298e75ed87571486f4364', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'qwen2', 'families': ['qwen2'], 'parameter_size': '7.6B', 'quantization_level': 'Q4_K_M', 'context_length': 32768, 'embedding_length': 3584, 'runner': 'ggml'}, 'capabilities': ['completion', 'tools', 'insert']}]}


In [38]:
response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "qwen2.5-coder:7b",
        "messages": [
            {
                "role": "user",
                "content": "Rewrite this product description concisely: Stainless steel water bottle, 750ml capacity, double-wall insulation."
            }
        ],
        "stream": False
    },
    timeout=180
)

response.raise_for_status()
print(response.json()["message"]["content"])

Stainless steel, 750ml, double-wall insulated water bottle.


In [39]:
print(items[0].full)

Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)
['From the Manufacturer', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we're the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]
['Interior half only', 'Requires F58 to complete handle set', 'Non handed knob style', '4" minimum center to center door prep required for this two piece model.', 'Lifetime Mechanical and Finish Warranty']
{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", "Handle Material": "Bronze", "Package Type": "Standard Packaging", "Unit Count": "1.0 Count", "Number of Items": "1", "Manufacturer": "Schlage", "Product Dimensions": "8.1 x 4

In [40]:
item = items[0]

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": item.full
    }
]

response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "qwen2.5-coder:7b",
        "messages": messages,
        "stream": False
    },
    timeout=180
)

response.raise_for_status()

clean_description = response.json()["message"]["content"]

print(clean_description)

Title: Schlage F59 & 613 Interior Knob with Deadbolt
Category: Hardware & Security
Brand: Schlage
Description: A sleek oil rubbed bronze interior knob and deadbolt set, perfect for securing your interior doors.
Details: Precision-engineered for security, featuring 100% solid brass construction and easy-to-install design.
